# Part 2b — NLP Feature Engineering with spaCy

> **Case study part:** Part 2 of 3 (second of two notebooks)
> **Required for:** All students (3-credit and 4-credit)
> **Role:** You are a Real Estate Analyst at Quad Capital Partners (QCP).

The Part 2a baseline used only structured tabular features. But every
listing also has a 100–500 word **description** written by the listing
agent — the field that brags about the granite countertops, the new roof,
the walkable distance to campus, and the basement that's "perfect for a
graduate-student rental."

In this notebook we use **spaCy** to turn that free text into structured
features and feed them back into the same modeling pipeline from Part 2a.
The goal is a measurable, honest improvement on the same held-out test set.


## ⚙️ Setup

We need spaCy and its small English model. If the model is not yet installed,
the next cell will download it.


In [ ]:
# Run once per machine if needed:
# !pip install spacy
# !python -m spacy download en_core_web_sm


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import re
from collections import Counter

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

import spacy
from spacy.matcher import PhraseMatcher

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 80)


In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
print("spaCy model loaded:", nlp.meta["name"], nlp.meta["version"])


:::{note} Why disable `parser` and `ner`?

We do not need dependency parses or named-entity recognition for this task,
and disabling them roughly **3× speeds up** processing 35,000 descriptions.
We keep the tokenizer, the part-of-speech tagger, and the lemmatizer — that
is enough for keyword matching, lemmatization, and noun-chunk extraction with
a custom rule.
:::


## 📥 Load the Cleaned Data and the Part 2a Split


In [ ]:
DATA_DIR = "data"

df = pd.read_parquet(f"{DATA_DIR}/properties_clean.parquet")
splits = pd.read_parquet(f"{DATA_DIR}/baseline_split.parquet")

train_idx = splits.loc[splits["split"] == "train", "row_index"].values
test_idx  = splits.loc[splits["split"] == "test",  "row_index"].values

print(f"{len(df):,} listings, {len(train_idx):,} train, {len(test_idx):,} test")


## 🔎 A Quick Look at What Listing Agents Actually Write


In [ ]:
sample = df.dropna(subset=["description"]).sample(n=3, random_state=7)
for _, row in sample.iterrows():
    print(f"--- {row['address_street']}, {row['market']}  (${row['price']:,}) ---")
    print(row["description"][:400], "...\n")


## 🧰 spaCy Preprocessing

For each description we want:

1. A **clean lemma string** with stopwords and punctuation removed (good for
   TF-IDF), and
2. A list of **noun chunks** (good for surfacing what the listing emphasizes —
   "granite countertops," "finished basement," "walking distance to campus").

We process descriptions with `nlp.pipe(...)` in batches because that is much
faster than calling `nlp(text)` in a Python loop.


In [ ]:
def spacy_features(texts, batch_size=200):
    """Return a DataFrame with one row per input text."""
    out = []
    docs = nlp.pipe(texts, batch_size=batch_size)
    for doc in docs:
        lemmas = [
            t.lemma_.lower()
            for t in doc
            if not t.is_stop and not t.is_punct and not t.is_space and t.is_alpha
        ]
        # We re-enable noun-chunk-style extraction with a lightweight rule:
        # contiguous spans of ADJ*+NOUN+ that exclude stopwords.
        chunks = []
        current = []
        for t in doc:
            if t.pos_ in {"ADJ", "NOUN", "PROPN"} and not t.is_stop:
                current.append(t.lemma_.lower())
            else:
                if len(current) >= 2:
                    chunks.append(" ".join(current))
                current = []
        if len(current) >= 2:
            chunks.append(" ".join(current))

        out.append({
            "n_tokens": len(doc),
            "n_lemmas": len(lemmas),
            "lemma_text": " ".join(lemmas),
            "noun_chunks": chunks,
            "n_noun_chunks": len(chunks),
        })
    return pd.DataFrame(out)


In [ ]:
texts = df["description"].fillna("").tolist()
nlp_feats = spacy_features(texts)
nlp_feats.head(3)


## 🏷️ Amenity Detection with `PhraseMatcher`

Listing agents reuse a small vocabulary of "selling words." We define an
amenity dictionary, compile it into a spaCy `PhraseMatcher`, and emit a
**binary flag per amenity** for every listing. These are the kinds of
features a hedonic pricing model loves.


In [ ]:
AMENITIES = {
    "granite":          ["granite countertop", "granite counter"],
    "stainless":        ["stainless steel appliance", "stainless appliance"],
    "hardwood":         ["hardwood floor", "hardwood flooring"],
    "updated_kitchen":  ["updated kitchen", "renovated kitchen", "remodeled kitchen", "new kitchen"],
    "open_floor_plan":  ["open floor plan", "open concept"],
    "primary_suite":    ["primary suite", "master suite", "owner suite"],
    "finished_basement":["finished basement", "walkout basement"],
    "two_car_garage":   ["two car garage", "2 car garage", "two-car garage"],
    "fenced_yard":      ["fenced yard", "fenced backyard", "privacy fence"],
    "deck_or_patio":    ["deck", "patio", "screened porch"],
    "new_roof":         ["new roof", "newer roof", "roof replaced"],
    "new_hvac":         ["new hvac", "new furnace", "new ac", "new air conditioner"],
    "near_campus":      ["walking distance to campus", "close to campus", "near campus", "minutes to campus"],
    "investor":         ["investor", "rental property", "investment property", "rental income"],
    "fixer_upper":      ["fixer upper", "needs work", "as is", "tlc"],
    "luxury":           ["luxury", "high end", "designer", "custom built"],
}

matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
for label, phrases in AMENITIES.items():
    matcher.add(label, [nlp.make_doc(p) for p in phrases])

def amenity_flags(texts, batch_size=200):
    rows = []
    docs = nlp.pipe(texts, batch_size=batch_size, disable=["tagger", "lemmatizer"])
    label_lookup = {nlp.vocab.strings[label]: label for label in AMENITIES}
    for doc in docs:
        hits = {f"amen_{label}": 0 for label in AMENITIES}
        for match_id, _, _ in matcher(doc):
            hits[f"amen_{label_lookup[match_id]}"] = 1
        rows.append(hits)
    return pd.DataFrame(rows)


In [ ]:
amen = amenity_flags(texts)
amen.head(3)


In [ ]:
amen_share = amen.mean().sort_values(ascending=False).mul(100).round(1)
amen_share.to_frame("pct_listings_mentioning")


In [ ]:
fig = px.bar(
    amen_share.sort_values().reset_index().rename(
        columns={"index": "amenity", 0: "pct"}),
    x=0, y="index", orientation="h",
    title="Share of campus-town listings mentioning each amenity",
    labels={"0": "% of listings", "index": ""},
    height=500,
)
fig.show()


## 💵 Do Amenity Mentions Move Price?

A quick before-modeling sanity check: average price for listings that mention
each amenity vs. listings that do not.


In [ ]:
combo = pd.concat([df.reset_index(drop=True)[["price"]], amen.reset_index(drop=True)], axis=1)
lift = []
for c in amen.columns:
    with_a    = combo.loc[combo[c] == 1, "price"].median()
    without_a = combo.loc[combo[c] == 0, "price"].median()
    lift.append({
        "amenity": c.replace("amen_", ""),
        "median_with": with_a,
        "median_without": without_a,
        "pct_lift": (with_a - without_a) / without_a * 100,
    })
lift_df = pd.DataFrame(lift).sort_values("pct_lift", ascending=False)
lift_df


In [ ]:
fig = px.bar(
    lift_df.sort_values("pct_lift"),
    x="pct_lift", y="amenity", orientation="h",
    color="pct_lift", color_continuous_scale="RdBu", color_continuous_midpoint=0,
    title="Naive price lift when each amenity is mentioned (median, %)",
    labels={"pct_lift": "Median price lift vs. listings without the phrase",
            "amenity": ""},
    height=500,
)
fig.show()


:::{caution} Confounded, but useful

This chart is *correlational*. "Luxury" lifts the price 60% partly because
genuinely expensive houses are described that way — not because adding the
word makes a house worth more. The proper test is whether these features
improve the *predictive* model, which is what we do next.
:::


## 🧱 Build the NLP-Augmented Modeling Dataset


In [ ]:
# Combine cleaned tabular features with spaCy-derived features.
features_full = pd.concat(
    [df.reset_index(drop=True), nlp_feats, amen.reset_index(drop=True)],
    axis=1,
)
features_full["log_price"] = np.log1p(features_full["price"])

NUMERIC_FEATURES = [
    "beds", "baths", "living_area", "lot_size",
    "year_built", "reso_parking_capacity",
    "latitude", "longitude",
    # New: NLP-derived numeric signals
    "n_tokens", "n_noun_chunks",
] + list(amen.columns)

CATEGORICAL_FEATURES = [
    "address_state", "address_city", "address_zipcode",
    "reso_architectural_style",
]
TEXT_FEATURE = "lemma_text"

X = features_full[NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TEXT_FEATURE]].copy()
X["address_zipcode"] = X["address_zipcode"].astype(str)
X[TEXT_FEATURE] = X[TEXT_FEATURE].fillna("")

y_log = features_full["log_price"]
y_raw = features_full["price"]

# Use the same split as Part 2a so we are comparing apples to apples.
X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train_log, y_test_log = y_log.loc[train_idx], y_log.loc[test_idx]
y_train_raw, y_test_raw = y_raw.loc[train_idx], y_raw.loc[test_idx]
print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")


## 🔧 Pipeline = Tabular + TF-IDF on Description Lemmas


In [ ]:
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
])
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
])
text_pipeline = TfidfVectorizer(
    max_features=2_000, ngram_range=(1, 2), min_df=10,
)

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, NUMERIC_FEATURES),
    ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
    ("txt", text_pipeline, TEXT_FEATURE),
])

nlp_model = Pipeline([
    ("preprocess", preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=400, max_depth=4, learning_rate=0.05, random_state=42,
    )),
])

nlp_model.fit(X_train, y_train_log)


## 📏 Compare Against the Baseline on the Same Test Set


In [ ]:
def metrics(y_true, y_pred):
    return {
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2":   r2_score(y_true, y_pred),
        "MAPE": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
    }

# NLP-augmented predictions
nlp_pred_test = np.expm1(nlp_model.predict(X_test))

# Re-train the baseline on identical splits for an apples-to-apples table.
baseline_pre = ColumnTransformer([
    ("num", numeric_pipeline, [c for c in NUMERIC_FEATURES if not c.startswith("amen_") and c not in {"n_tokens", "n_noun_chunks"}]),
    ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
])
baseline_model = Pipeline([
    ("preprocess", baseline_pre),
    ("regressor", GradientBoostingRegressor(
        n_estimators=400, max_depth=4, learning_rate=0.05, random_state=42,
    )),
])
baseline_model.fit(X_train, y_train_log)
baseline_pred_test = np.expm1(baseline_model.predict(X_test))

comparison = pd.DataFrame({
    "Baseline (Part 2a)": metrics(y_test_raw, baseline_pred_test),
    "Baseline + spaCy NLP": metrics(y_test_raw, nlp_pred_test),
}).T
comparison


:::{tip} Read the table the way the Investment Committee will

The metric they care about is **MAPE**. A drop of even one percentage point
across thousands of listings translates into a real dollar improvement in
acquisition decisions. Always quote the *change* alongside the absolute
number ("MAPE fell from X% to Y%, a Z% improvement").
:::


## 🧪 Where Did NLP Help Most?

We expect NLP to help the most where the tabular features alone are weakest —
typically in mid-priced listings whose value depends on condition and
amenities, not just size and location.


In [ ]:
diag = pd.DataFrame({
    "actual": y_test_raw.values,
    "baseline": baseline_pred_test,
    "nlp": nlp_pred_test,
    "market": features_full.loc[test_idx, "market"].values,
})
diag["baseline_pct_err"] = (diag["actual"] - diag["baseline"]).abs() / diag["actual"] * 100
diag["nlp_pct_err"]      = (diag["actual"] - diag["nlp"]).abs()      / diag["actual"] * 100
diag["improvement_pp"]   = diag["baseline_pct_err"] - diag["nlp_pct_err"]

by_market = (
    diag.groupby("market", as_index=False)
        .agg(n=("actual", "count"),
             mean_baseline=("baseline_pct_err", "mean"),
             mean_nlp=("nlp_pct_err", "mean"),
             improvement_pp=("improvement_pp", "mean"))
        .sort_values("improvement_pp", ascending=False)
)
by_market.round(2)


In [ ]:
fig = px.bar(
    by_market.sort_values("improvement_pp"),
    x="improvement_pp", y="market", orientation="h",
    color="improvement_pp", color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    title="Reduction in mean |% error| from adding NLP features (percentage points)",
    labels={"improvement_pp": "Δ MAPE (baseline − NLP), percentage points",
            "market": ""},
    height=600,
)
fig.show()


## 💾 Persist the NLP Features for Part 3

4-credit students will reuse `nlp_feats` and `amen` in Part 3 alongside the
price-history features. We save them to Parquet so Part 3 does not have to
re-run spaCy over 35,000 descriptions.


In [ ]:
nlp_pack = pd.concat(
    [df[["property_id"]].reset_index(drop=True),
     nlp_feats[["n_tokens", "n_noun_chunks"]].reset_index(drop=True),
     amen.reset_index(drop=True)],
    axis=1,
)
nlp_pack.to_parquet(f"{DATA_DIR}/nlp_features.parquet", index=False)
print(f"Saved {len(nlp_pack):,} rows of NLP features → {DATA_DIR}/nlp_features.parquet")


## 🧠 Investment-Committee Takeaways (Part 2b)

Suggested bullets for the slide deck:

- Adding NLP-derived features cut the AVM's MAPE from **X% to Y%** on the
  same held-out sample.
- The amenity flags with the largest naive price lift were `luxury`,
  `granite`, and `updated_kitchen`. After controlling for size and location,
  the model still gives them positive weight.
- The improvement is largest in **[markets]**, where listings are more
  heterogeneous and the description carries more signal.

3-credit students stop here. **4-credit students** continue to **Part 3**,
where we incorporate the property-level price-history event log to model
realized sale outcomes — not just list prices.
